# Running a Small LLM Locally

Modern language models in the 1B-7B parameter range can run on a single GPU with a few gigabytes of VRAM, which includes the free T4 available on Google Colab. This notebook walks through how to load one of these models, why quantization lets you fit larger models in less memory, and how to run inference with the HuggingFace `transformers` library. By the end you will have loaded a real model, queried it with three different kinds of prompts, and observed where it performs well and where it struggles.

In [ ]:
# Uncomment and run the cell below if you are on Colab or a fresh environment
# !pip install transformers accelerate bitsandbytes -q

import warnings
warnings.filterwarnings("ignore")

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline,
    TextGenerationPipeline,
)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. The Landscape of Sub-7B Models

Several open-weight models are small enough to run on consumer hardware while still being genuinely useful. Here is a quick comparison of the most commonly used ones as of 2025:

| Model | Params | Context Length | Strengths | Notes |
|---|---|---|---|---|
| `microsoft/phi-2` | 2.7B | 2048 tokens | Code, reasoning, compact size | Good baseline for experimentation |
| `microsoft/Phi-3-mini-4k-instruct` | 3.8B | 4096 tokens | Strong instruction following, reasoning | Punches above its weight class |
| `mistralai/Mistral-7B-Instruct-v0.3` | 7B | 32768 tokens | Long context, general purpose | Best-in-class at 7B for many tasks |
| `meta-llama/Llama-3.2-3B-Instruct` | 3B | 128000 tokens | Huge context, multilingual | Requires HuggingFace login |
| `Qwen/Qwen2.5-3B-Instruct` | 3B | 32768 tokens | Code, math, multilingual | Strong at structured output |
| `Qwen/Qwen2.5-1.5B-Instruct` | 1.5B | 32768 tokens | Tiny footprint, surprisingly capable | Best choice for Colab free tier |

**Choosing a model for this notebook:** We will use `Qwen/Qwen2.5-1.5B-Instruct` because it fits comfortably on a T4 even without quantization, and it follows instructions reliably. If you have more VRAM (e.g., a T4 with 15 GB free), try `Qwen/Qwen2.5-3B-Instruct` or `microsoft/phi-2` for a noticeable quality jump.

**A note on model IDs:** The `-Instruct` suffix means the model has been fine-tuned to follow instructions (system/user/assistant format). Base models without this suffix are pre-trained only and are harder to prompt effectively.

## 2. Quantization: Fitting More Model Into Less Memory

A language model is a large collection of floating-point numbers called weights. By default these are stored as 32-bit or 16-bit floats. Quantization stores them with fewer bits:

- **FP32 (full precision):** 4 bytes per weight. A 7B model uses ~28 GB.
- **FP16 / BF16 (half precision):** 2 bytes per weight. A 7B model uses ~14 GB.
- **INT8 (8-bit quantization):** ~1 byte per weight. A 7B model uses ~7 GB.
- **INT4 (4-bit quantization):** ~0.5 bytes per weight. A 7B model uses ~3.5 GB.

The tradeoff is quality: fewer bits means some precision is lost. In practice, 4-bit quantization with the NF4 format (used by `bitsandbytes`) causes very little degradation for most tasks.

The `bitsandbytes` library handles this transparently. You configure it with `BitsAndBytesConfig` and pass it to `from_pretrained`. The model loads in quantized form and never requires the full-precision weights to be in memory.

```
FP32 weights:   [0.3241, -0.1823, 0.7654, -0.2109, ...]
                 32 bits each

4-bit NF4:      [  3,      1,      14,      2,     ...]
                 4 bits each, mapped back to floats via a learned codebook
```

`device_map="auto"` tells `accelerate` to automatically distribute model layers across available devices (GPU first, then CPU, then disk if needed). On a single-GPU machine this simply puts everything on the GPU.

In [ ]:
# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",        # NF4 is better than standard int4 for LLMs
    bnb_4bit_compute_dtype=torch.bfloat16,  # compute in BF16 for speed + stability
    bnb_4bit_use_double_quant=True,   # quantize the quantization constants too (saves ~0.4 GB more)
)

print("BitsAndBytesConfig created:")
print(f"  load_in_4bit: {bnb_config.load_in_4bit}")
print(f"  quant_type: {bnb_config.bnb_4bit_quant_type}")
print(f"  compute_dtype: {bnb_config.bnb_4bit_compute_dtype}")
print(f"  double_quant: {bnb_config.bnb_4bit_use_double_quant}")

## 3. Loading the Model and Tokenizer

Every HuggingFace model has two components:

- **Tokenizer:** Converts raw text into token IDs (integers) and back. Different models use different tokenizers and vocabulary sizes.
- **Model:** The neural network itself, which takes token IDs and produces logits (scores for the next token).

For inference (not training), you typically want `torch_dtype=torch.bfloat16` or `float16` to halve memory usage with minimal quality loss. When using 4-bit quantization via `bitsandbytes`, the `quantization_config` argument handles memory reduction instead.

## 3a. Loading Llama-3.2-1B-Instruct in BF16

`meta-llama/Llama-3.2-1B-Instruct` is Meta's smallest Llama 3 model and fits comfortably in 2-3 GB of VRAM when loaded in `bfloat16`. Unlike the quantized path above, loading in BF16 preserves full half-precision weights and is simpler to debug.

Key points:
- `torch_dtype=torch.bfloat16` stores every weight as a 2-byte BF16 float (vs. 4-byte FP32).
- `device_map="auto"` from `accelerate` places all layers on the GPU automatically.
- You need a HuggingFace account and must accept the Llama 3 license at `hf.co/meta-llama/Llama-3.2-1B-Instruct` before the download will succeed. Log in with `huggingface-cli login` or set `HF_TOKEN` in your environment.

**Memory footprint:** we measure GPU memory before and after loading to see exactly how much the model occupies.

In [ ]:
# !pip install transformers accelerate -q
# Requires: huggingface-cli login  (or set HF_TOKEN env variable)
# You must accept the Llama 3 license at:
#   https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

LLAMA_MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

# --- Memory before loading ---
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    mem_before = torch.cuda.memory_allocated() / 1e9
    print(f"GPU memory before model load: {mem_before:.3f} GB")
else:
    print("No CUDA GPU detected. Model will load on CPU (slower).")

print(f"\nLoading tokenizer from {LLAMA_MODEL_ID} ...")
llama_tokenizer = AutoTokenizer.from_pretrained(LLAMA_MODEL_ID)

print(f"Loading model in bfloat16 with device_map='auto' ...")
llama_model = AutoModelForCausalLM.from_pretrained(
    LLAMA_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# --- Memory after loading ---
if torch.cuda.is_available():
    mem_after = torch.cuda.memory_allocated() / 1e9
    mem_peak  = torch.cuda.max_memory_allocated() / 1e9
    print(f"\nGPU memory after model load:  {mem_after:.3f} GB")
    print(f"GPU peak memory allocated:    {mem_peak:.3f} GB")
    print(f"Memory used by model:         {mem_after - mem_before:.3f} GB")

total_params = sum(p.numel() for p in llama_model.parameters())
print(f"\nTotal parameters: {total_params / 1e9:.2f}B")
print(f"dtype: {next(llama_model.parameters()).dtype}")
print(f"Device map: {llama_model.hf_device_map}" if hasattr(llama_model, "hf_device_map") else "")

## 3b. Greedy Decoding: Calling `model.generate` Directly

Using `pipeline` is convenient, but calling `model.generate` directly gives you full control. Greedy decoding (`do_sample=False`) always picks the single highest-probability next token at each step. It is deterministic and fast, but can produce repetitive or locally optimal (not globally optimal) sequences.

Steps:
1. Tokenize the prompt and move tensors to the model's device.
2. Call `model.generate` with `do_sample=False`.
3. Slice off the prompt tokens from the output (the model echoes the input).
4. Decode back to a string with `tokenizer.decode`.

In [ ]:
import torch

# Format the prompt using the Llama 3 chat template
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user",   "content": "Explain what a transformer neural network is in two sentences."},
]

prompt_text = llama_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

# Tokenize and move to the same device as the model
inputs = llama_tokenizer(prompt_text, return_tensors="pt")
input_ids = inputs["input_ids"].to(llama_model.device)
attention_mask = inputs["attention_mask"].to(llama_model.device)

print(f"Prompt token count: {input_ids.shape[1]}")

# Greedy decoding: do_sample=False means always pick the argmax token
with torch.no_grad():
    output_ids = llama_model.generate(
        input_ids,
        attention_mask=attention_mask,
        max_new_tokens=128,
        do_sample=False,          # greedy
        pad_token_id=llama_tokenizer.eos_token_id,
    )

# output_ids includes the prompt; slice it off
new_token_ids = output_ids[0, input_ids.shape[1]:]
decoded = llama_tokenizer.decode(new_token_ids, skip_special_tokens=True)

print(f"Generated {len(new_token_ids)} new tokens\n")
print("=" * 60)
print("Greedy output:")
print(decoded)

## 3c. Sampling Parameters: Temperature Sweep

Temperature scales the logits before the softmax that produces the probability distribution over the vocabulary. Intuitively:

- `temperature < 1.0`: sharpens the distribution (model is more confident, less varied).
- `temperature = 1.0`: leaves the distribution unchanged.
- `temperature > 1.0`: flattens the distribution (model is more adventurous, can produce unusual words).

We run the same creative prompt at four temperatures and compare how diversity changes.

In [ ]:
import torch

sweep_prompt_messages = [
    {"role": "system", "content": "You are a creative writer."},
    {"role": "user",   "content": "Write a single sentence describing the ocean at night."},
]

sweep_prompt_text = llama_tokenizer.apply_chat_template(
    sweep_prompt_messages,
    tokenize=False,
    add_generation_prompt=True,
)
sweep_inputs = llama_tokenizer(sweep_prompt_text, return_tensors="pt")
sweep_input_ids = sweep_inputs["input_ids"].to(llama_model.device)
sweep_attention_mask = sweep_inputs["attention_mask"].to(llama_model.device)

temperatures = [0.1, 0.7, 1.0, 1.5]

print("Same prompt, four temperatures (run each 2x to see variance):\n")
for temp in temperatures:
    outputs_for_temp = []
    for run in range(2):
        with torch.no_grad():
            out = llama_model.generate(
                sweep_input_ids,
                attention_mask=sweep_attention_mask,
                max_new_tokens=60,
                do_sample=True,
                temperature=temp,
                pad_token_id=llama_tokenizer.eos_token_id,
            )
        new_ids = out[0, sweep_input_ids.shape[1]:]
        text = llama_tokenizer.decode(new_ids, skip_special_tokens=True).strip()
        outputs_for_temp.append(text)

    print(f"--- temperature={temp} ---")
    for i, t in enumerate(outputs_for_temp, 1):
        print(f"  Run {i}: {t}")
    print()

print("Observations to notice:")
print("  temp=0.1 -> runs are nearly identical, safe/predictable word choices")
print("  temp=0.7 -> small variation between runs, still coherent")
print("  temp=1.0 -> noticeable variation, occasionally unusual phrasing")
print("  temp=1.5 -> high variation, sometimes grammatically odd or surreal")

## 3d. Top-p and Top-k Sampling

Temperature controls how spread out the distribution is, but you can also directly limit *which* tokens are eligible for sampling:

- **Top-k (`top_k=50`):** At each step, keep only the 50 most probable tokens and redistribute probability mass among them. Tokens ranked 51+ are set to zero probability.
- **Top-p / nucleus sampling (`top_p=0.9`):** Keep the smallest set of tokens whose cumulative probability exceeds 0.9. This is adaptive: if one token dominates (probability 0.95), only that token is kept; if many tokens share probability, more are kept.

In practice, `top_p=0.9` is more robust than a fixed `top_k` because it adapts to the shape of each distribution. Combining `top_p=0.9` with `temperature=0.7` is a common production default.

In [ ]:
import torch

sampling_prompt_messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user",   "content": "What are three things you could do on a rainy afternoon?"},
]
sampling_prompt_text = llama_tokenizer.apply_chat_template(
    sampling_prompt_messages,
    tokenize=False,
    add_generation_prompt=True,
)
s_inputs = llama_tokenizer(sampling_prompt_text, return_tensors="pt")
s_input_ids = s_inputs["input_ids"].to(llama_model.device)
s_attn_mask = s_inputs["attention_mask"].to(llama_model.device)


def sample_with_params(input_ids, attn_mask, label, **kwargs):
    with torch.no_grad():
        out = llama_model.generate(
            input_ids,
            attention_mask=attn_mask,
            max_new_tokens=100,
            do_sample=True,
            pad_token_id=llama_tokenizer.eos_token_id,
            **kwargs,
        )
    new_ids = out[0, input_ids.shape[1]:]
    text = llama_tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    print(f"\n--- {label} ---")
    print(text)


# Baseline: temperature only
sample_with_params(s_input_ids, s_attn_mask, "temperature=0.8 (no top_p/top_k)", temperature=0.8)

# top_p nucleus sampling
sample_with_params(s_input_ids, s_attn_mask, "top_p=0.9, temperature=0.8", temperature=0.8, top_p=0.9)

# top_k sampling
sample_with_params(s_input_ids, s_attn_mask, "top_k=50, temperature=0.8", temperature=0.8, top_k=50)

# Combined top_p + top_k (common production default)
sample_with_params(s_input_ids, s_attn_mask, "top_p=0.9, top_k=50, temperature=0.8",
                   temperature=0.8, top_p=0.9, top_k=50)

print("\nNote: top_p and top_k both reduce the risk of very low-probability")
print("'surprise' tokens. The output variety you observe depends on the")
print("specific token distribution at each step.")

## 3e. Token Probability Visualization

Every forward pass through the model produces a vector of logits with one value per vocabulary token. Converting these to probabilities with softmax shows which tokens the model considers for the *next* position.

Visualizing the top-20 probabilities reveals:
- How confident the model is (one token dominating vs. many sharing probability).
- Whether the distribution is flat (high temperature regime) or sharp (low temperature or clear context).

We use a single forward pass (no generation) to get the logits at the last prompt position.

In [ ]:
import torch
import matplotlib.pyplot as plt

# Prompt that has a fairly predictable next token ("Paris")
viz_messages = [
    {"role": "user", "content": "The capital of France is"},
]
viz_prompt = llama_tokenizer.apply_chat_template(
    viz_messages,
    tokenize=False,
    add_generation_prompt=True,
)
# Strip the trailing assistant-start token so the model predicts the next real word
viz_inputs = llama_tokenizer(viz_prompt, return_tensors="pt")
viz_input_ids = viz_inputs["input_ids"].to(llama_model.device)

# Single forward pass to get logits (no generation)
with torch.no_grad():
    outputs = llama_model(viz_input_ids)

# outputs.logits shape: (batch=1, seq_len, vocab_size)
# We want the logits at the LAST token position (what comes next)
last_logits = outputs.logits[0, -1, :]           # (vocab_size,)
probs = torch.softmax(last_logits, dim=-1)

# Extract top-20 token IDs and their probabilities
top_k_vals, top_k_ids = torch.topk(probs, k=20)
top_k_vals  = top_k_vals.cpu().float().numpy()
top_k_ids   = top_k_ids.cpu().numpy()

# Decode each token ID to a readable string
top_k_tokens = [llama_tokenizer.decode([tid]).strip() or f"[{tid}]" for tid in top_k_ids]

# Plot
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(range(20), top_k_vals[::-1], color="steelblue")
ax.set_yticks(range(20))
ax.set_yticklabels(top_k_tokens[::-1], fontsize=10)
ax.set_xlabel("Probability")
ax.set_title('Top-20 next-token probabilities for "The capital of France is"')
ax.axvline(x=0, color="black", linewidth=0.5)

for bar, val in zip(bars, top_k_vals[::-1]):
    ax.text(val + 0.002, bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}", va="center", fontsize=8)

plt.tight_layout()
plt.show()

print(f"\nTop token: '{top_k_tokens[0]}' with probability {top_k_vals[0]:.4f}")
print(f"Sum of top-20 probabilities: {top_k_vals.sum():.4f}")
print("A sharp distribution (one token has very high prob) indicates the model is confident.")

## 3f. Full RAG Pipeline

RAG (Retrieval-Augmented Generation) grounds the model in real documents instead of relying on parametric memory. The pipeline has three stages:

1. **Embed the corpus:** Convert each document into a dense vector using a sentence embedding model.
2. **Index and retrieve:** Build a FAISS index, then for each query find the top-k most similar document vectors.
3. **Generate with context:** Inject the retrieved documents into the prompt and ask the model to answer.

We use `BAAI/bge-small-en-v1.5` for embeddings (it is small, fast, and performs well on English retrieval benchmarks) and FAISS `IndexFlatIP` with normalized vectors for exact inner-product (= cosine) search.

In [ ]:
# !pip install sentence-transformers faiss-cpu -q

import numpy as np
import torch
from sentence_transformers import SentenceTransformer
import faiss

# --- Small document corpus ---
CORPUS = [
    {"title": "Photosynthesis Basics",
     "text": "Photosynthesis is the process by which plants use sunlight, water, and carbon dioxide to produce glucose and oxygen. It occurs in the chloroplasts."},
    {"title": "The Water Cycle",
     "text": "The water cycle describes the continuous movement of water through evaporation, condensation, precipitation, and collection back into bodies of water."},
    {"title": "Newton's Laws",
     "text": "Newton's second law states that force equals mass times acceleration (F = ma). His first law says objects remain at rest or in motion unless acted upon by a force."},
    {"title": "DNA and Genetics",
     "text": "DNA (deoxyribonucleic acid) carries genetic information. The double helix structure was discovered by Watson and Crick in 1953 using X-ray data from Franklin."},
    {"title": "Machine Learning Overview",
     "text": "Machine learning is a subset of AI where systems learn patterns from data without being explicitly programmed. Common types include supervised, unsupervised, and reinforcement learning."},
    {"title": "The Solar System",
     "text": "Our solar system contains eight planets orbiting the Sun. Jupiter is the largest; Mercury is the closest to the Sun. Earth is the only planet known to support life."},
    {"title": "Quantum Mechanics",
     "text": "Quantum mechanics describes physical phenomena at atomic scales. Key principles include wave-particle duality, the uncertainty principle, and superposition of states."},
    {"title": "Climate Change",
     "text": "Climate change refers to long-term shifts in global temperatures and weather patterns, largely driven since the industrial era by burning fossil fuels and releasing CO2."},
]

# 1. Embed the corpus
print("Loading embedding model: BAAI/bge-small-en-v1.5 ...")
embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

corpus_texts = [doc["text"] for doc in CORPUS]
corpus_embeddings = embed_model.encode(corpus_texts, normalize_embeddings=True, show_progress_bar=True)
print(f"\nCorpus embeddings shape: {corpus_embeddings.shape}")  # (8, 384)

# 2. Build FAISS index (IndexFlatIP = exact inner product = cosine with normalized vectors)
dim = corpus_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(corpus_embeddings.astype(np.float32))
print(f"FAISS index built. Total vectors indexed: {index.ntotal}")

# 3. Retrieval function
def retrieve(query: str, k: int = 3):
    query_vec = embed_model.encode([query], normalize_embeddings=True)
    scores, indices = index.search(query_vec.astype(np.float32), k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({"title": CORPUS[idx]["title"],
                         "text": CORPUS[idx]["text"],
                         "score": float(score)})
    return results

# 4. RAG generation function
def rag_answer(query: str, k: int = 3) -> str:
    retrieved_docs = retrieve(query, k=k)

    # Build context string from top-k docs
    context_parts = []
    for i, doc in enumerate(retrieved_docs, 1):
        context_parts.append(f"[Doc {i}: {doc['title']}]\n{doc['text']}")
    context = "\n\n".join(context_parts)

    messages = [
        {"role": "system", "content": (
            "You are a helpful assistant. Answer the user's question using only "
            "the provided context documents. If the answer is not in the context, say so."
        )},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"},
    ]
    prompt = llama_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = llama_tokenizer(prompt, return_tensors="pt")
    input_ids = inputs["input_ids"].to(llama_model.device)
    attn_mask = inputs["attention_mask"].to(llama_model.device)

    with torch.no_grad():
        out = llama_model.generate(
            input_ids,
            attention_mask=attn_mask,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=llama_tokenizer.eos_token_id,
        )
    new_ids = out[0, input_ids.shape[1]:]
    return llama_tokenizer.decode(new_ids, skip_special_tokens=True).strip()

# --- Test the RAG pipeline ---
test_queries = [
    "How do plants make food?",
    "What is the largest planet in our solar system?",
    "Explain machine learning in simple terms.",
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    docs = retrieve(query, k=3)
    print("Retrieved docs:", [d["title"] for d in docs])
    answer = rag_answer(query, k=3)
    print(f"Answer: {answer}")

## 3g. Streaming Generation with `TextStreamer`

By default, `model.generate` buffers all output and returns it only when generation is complete. For long outputs this means a long wait with nothing on screen.

`TextStreamer` from `transformers` hooks into the generation loop and prints each decoded token to stdout as it is produced. This gives the familiar "typing" effect seen in chat interfaces. It requires no changes to the model or generation parameters -- just pass `streamer=streamer` to `generate`.

In [ ]:
import torch
from transformers import TextStreamer

stream_messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user",   "content": "Explain how FAISS enables fast nearest-neighbor search in dense vector spaces. Use 3-4 sentences."},
]
stream_prompt = llama_tokenizer.apply_chat_template(
    stream_messages,
    tokenize=False,
    add_generation_prompt=True,
)
stream_inputs = llama_tokenizer(stream_prompt, return_tensors="pt")
stream_input_ids  = stream_inputs["input_ids"].to(llama_model.device)
stream_attn_mask  = stream_inputs["attention_mask"].to(llama_model.device)

# TextStreamer prints tokens to stdout as they are generated.
# skip_prompt=True suppresses echoing the input prompt.
streamer = TextStreamer(llama_tokenizer, skip_prompt=True, skip_special_tokens=True)

print("Streaming output (tokens appear as generated):")
print("-" * 60)
with torch.no_grad():
    llama_model.generate(
        stream_input_ids,
        attention_mask=stream_attn_mask,
        max_new_tokens=150,
        do_sample=False,
        streamer=streamer,
        pad_token_id=llama_tokenizer.eos_token_id,
    )
print("\n" + "-" * 60)
print("Generation complete.")

## Exercise: RAG with Source Citation

The RAG pipeline above retrieves documents and uses them for answering, but the model does not explicitly mention which document it used. In real applications, citing sources builds user trust and makes it easier to verify answers.

Your task: modify the RAG generation prompt so that:
1. The system message instructs the model to include the source document title in its answer.
2. After generating the answer, verify that the model's output contains the title of at least one retrieved document.

The `CORPUS`, `retrieve()`, and `rag_answer()` functions from the RAG pipeline cell above are available.

In [ ]:
import torch

def rag_answer_with_citation(query: str, k: int = 3) -> str:
    """
    Retrieve top-k documents and generate an answer that cites the source title(s).

    Requirements:
    - The system message must ask the model to include the document title in the answer.
    - Return the generated answer string.
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Verification: run the citation-aware RAG and check that a document title appears
exercise_query = "What is the structure of DNA?"
answer_with_citation = rag_answer_with_citation(exercise_query, k=3)

print(f"Query: {exercise_query}")
print(f"\nAnswer:\n{answer_with_citation}")

# Auto-check: does the answer mention at least one retrieved document title?
retrieved = retrieve(exercise_query, k=3)
retrieved_titles = [doc["title"] for doc in retrieved]
print(f"\nRetrieved document titles: {retrieved_titles}")

cited_any = any(title.lower() in answer_with_citation.lower() for title in retrieved_titles)
print(f"\nCitation check passed: {cited_any}")
if not cited_any:
    print("The answer does not mention any retrieved document title.")
    print("Adjust your system prompt to instruct the model more explicitly.")

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
# Alternative (slightly larger, better quality): "microsoft/phi-2"
# Alternative (even larger, best quality at this size): "Qwen/Qwen2.5-3B-Instruct"

print(f"Loading tokenizer from {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

print(f"Loading model with 4-bit quantization...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",           # automatically place layers on GPU/CPU
    trust_remote_code=True,
)

print("\nModel loaded.")
print(f"Model device map: {model.hf_device_map}" if hasattr(model, 'hf_device_map') else "")

# Report memory usage
if torch.cuda.is_available():
    used_gb = torch.cuda.memory_allocated() / 1e9
    reserved_gb = torch.cuda.memory_reserved() / 1e9
    print(f"GPU memory allocated: {used_gb:.2f} GB")
    print(f"GPU memory reserved:  {reserved_gb:.2f} GB")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params / 1e9:.2f}B")

## 4. Running Inference with `pipeline`

The `transformers` `pipeline` function wraps the tokenizer + model into a single callable. For text generation, the key parameters are:

| Parameter | What it controls | Typical values |
|---|---|---|
| `max_new_tokens` | Maximum tokens to generate (not counting the prompt) | 64-512 |
| `temperature` | Randomness of sampling. Higher = more creative, lower = more deterministic | 0.1-1.0 |
| `do_sample` | Whether to sample (True) or always pick the most likely token (False) | True for creative tasks, False for factual |
| `top_p` | Nucleus sampling: only sample from the top tokens whose cumulative probability exceeds `top_p` | 0.9-0.95 |
| `repetition_penalty` | Penalize tokens that have already appeared. Helps avoid loops | 1.1-1.3 |

**Instruct models expect a specific prompt format.** Qwen2.5-Instruct uses the ChatML format:

```
<|im_start|>system
You are a helpful assistant.
<|im_end|>
<|im_start|>user
What is the capital of France?
<|im_end|>
<|im_start|>assistant
```

The tokenizer's `apply_chat_template` method handles this formatting automatically.

In [ ]:
# Create the pipeline
# We pass the already-loaded model so it doesn't reload from disk
generator: TextGenerationPipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

def chat(user_message: str, system_message: str = "You are a helpful assistant.",
         max_new_tokens: int = 256, temperature: float = 0.7) -> str:
    """Format a message using the model's chat template and run inference."""
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user",   "content": user_message},
    ]
    # apply_chat_template converts the message list to the model's expected string format
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,  # adds the <|im_start|>assistant prefix
    )
    outputs = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=temperature > 0,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
    )
    # The pipeline returns the full text including the prompt; strip the prompt part
    full_text = outputs[0]["generated_text"]
    # Extract just the assistant's response
    if "<|im_start|>assistant" in full_text:
        response = full_text.split("<|im_start|>assistant")[-1].strip()
        response = response.replace("<|im_end|>", "").strip()
    else:
        response = full_text[len(prompt):].strip()
    return response

print("Pipeline ready. Running a quick smoke test...")
test = chat("Say hello in one sentence.", max_new_tokens=32)
print(f"Model says: {test}")

## 5. Hands-On: Three Types of Prompts

We will test the model on three qualitatively different tasks:

1. **Factual recall:** Something with a clear correct answer, mostly in the model's training data.
2. **Multi-step reasoning:** A problem requiring the model to chain several steps.
3. **Code generation:** Writing a short Python function from a description.

For each, we will note where the model succeeds and where it falls short.

In [ ]:
# --- Prompt 1: Factual question ---
factual_prompt = """What are transformer neural networks, and why did they replace recurrent neural networks (RNNs) for most NLP tasks? Answer in 3-4 sentences."""

print("=" * 60)
print("PROMPT 1: Factual Recall")
print("=" * 60)
print(f"Question: {factual_prompt}\n")

factual_response = chat(
    factual_prompt,
    system_message="You are a knowledgeable AI researcher. Give accurate, concise answers.",
    max_new_tokens=200,
    temperature=0.3,  # low temperature for factual tasks
)
print(f"Response:\n{factual_response}")

print("\n--- Observation ---")
print("Small models generally do well on factual questions about topics")
print("that were well-represented in training data (ML, Python, general knowledge).")
print("They may confuse dates or specific details from less common domains.")

In [ ]:
# --- Prompt 2: Multi-step reasoning ---
reasoning_prompt = """A train leaves Station A at 9:00 AM traveling at 60 km/h toward Station B.
Another train leaves Station B at 10:00 AM traveling at 80 km/h toward Station A.
The distance between the stations is 280 km.

At what time do the trains meet? Show your reasoning step by step."""

print("=" * 60)
print("PROMPT 2: Multi-Step Reasoning")
print("=" * 60)
print(f"Question: {reasoning_prompt}\n")

reasoning_response = chat(
    reasoning_prompt,
    system_message="You are a careful reasoner. Work through problems step by step.",
    max_new_tokens=400,
    temperature=0.1,  # very low temperature for deterministic reasoning
)
print(f"Response:\n{reasoning_response}")

print("\n--- Expected Answer ---")
print("After 1 hour, Train A has traveled 60 km, so 220 km remain.")
print("Combined speed is 60 + 80 = 140 km/h.")
print("Time to close 220 km: 220 / 140 = 1.57 hours ≈ 1 hour 34 minutes.")
print("Trains meet at approximately 11:34 AM.")
print("\n--- Observation ---")
print("1.5B parameter models often make arithmetic errors or lose track of")
print("intermediate values in multi-step problems. Larger models (7B+) are")
print("more reliable here. Chain-of-thought prompting helps, but doesn't fix everything.")

In [ ]:
# --- Prompt 3: Code generation ---
code_prompt = """Write a Python function called `sliding_window_max` that takes a list of integers
and a window size k, and returns a list of the maximum value in each sliding window of size k.

Example: sliding_window_max([1, 3, -1, -3, 5, 3, 6, 7], k=3) should return [3, 3, 5, 5, 6, 7]

Include a docstring and a few test assertions."""

print("=" * 60)
print("PROMPT 3: Code Generation")
print("=" * 60)
print(f"Question: {code_prompt}\n")

code_response = chat(
    code_prompt,
    system_message="You are an expert Python programmer. Write clean, correct code.",
    max_new_tokens=400,
    temperature=0.2,  # low temperature: we want correct code, not creative variations
)
print(f"Response:\n{code_response}")

print("\n--- Observation ---")
print("Qwen2.5 models are notably good at code. Even the 1.5B version handles")
print("standard algorithmic problems well, though it may produce O(n*k) solutions")
print("rather than the optimal O(n) deque-based approach.")

### Trying to Execute the Generated Code

One advantage of notebooks is that we can immediately test whether the generated code actually runs correctly.

In [ ]:
# Attempt to extract and execute the generated function
import re

# Pull out the first Python code block from the response
code_blocks = re.findall(r'```python\n(.*?)```', code_response, re.DOTALL)
if not code_blocks:
    # Sometimes models omit the language tag
    code_blocks = re.findall(r'```\n(.*?)```', code_response, re.DOTALL)

if code_blocks:
    extracted_code = code_blocks[0]
    print("Extracted code block:")
    print("-" * 40)
    print(extracted_code)
    print("-" * 40)
    print("\nExecuting...")
    try:
        exec(extracted_code, globals())
        # Test it manually regardless of what assertions the model wrote
        result = sliding_window_max([1, 3, -1, -3, 5, 3, 6, 7], k=3)  # type: ignore
        expected = [3, 3, 5, 5, 6, 7]
        print(f"\nTest: sliding_window_max([1, 3, -1, -3, 5, 3, 6, 7], k=3)")
        print(f"Got:      {result}")
        print(f"Expected: {expected}")
        print(f"Correct:  {result == expected}")
    except Exception as e:
        print(f"Execution failed: {e}")
        print("This is common with 1.5B models -- the logic may be slightly off.")
        print("Try re-running the code generation cell with a different temperature.")
else:
    print("No code block found in the response. The model may have given prose instead.")
    print("Try running the code generation cell again.")

## 6. Temperature Effects

Temperature is the most important inference parameter to understand. Let's see it in action by running the same prompt at different temperatures.

In [ ]:
creative_prompt = "Describe a sunset in two sentences."

temperatures = [0.1, 0.7, 1.5]

for temp in temperatures:
    print(f"\n--- Temperature = {temp} ---")
    response = chat(
        creative_prompt,
        max_new_tokens=80,
        temperature=temp,
    )
    print(response)

print("\n" + "=" * 60)
print("Pattern to notice:")
print("  temp=0.1 -> almost the same output every run, predictable word choices")
print("  temp=0.7 -> natural variation, usually coherent")
print("  temp=1.5 -> more varied, sometimes surreal or grammatically odd")

## 7. Exercise

Work through all three parts below. For each, modify the `chat()` call in the corresponding code cell.

**Part A: System prompt engineering**
Change the system message to make the model respond like a pirate. Does the style change consistently? Does the factual content stay accurate?

**Part B: Length control**
Ask the model a question that typically gets a long answer (e.g., "Explain how gradient descent works"). Run it with `max_new_tokens=50` and `max_new_tokens=300`. Does the model gracefully truncate, or does it cut off mid-sentence?

**Part C: Find a failure mode**
Try to find a question where the 1.5B model gives a clearly wrong answer. Good candidates:
- A math problem requiring more than 3-4 arithmetic steps
- A question about a very recent event (after the model's knowledge cutoff)
- A question involving spatial reasoning (e.g., "If I face north and turn right twice, which direction am I facing?")

Document the failure and explain why you think the model got it wrong.

In [ ]:
# Part A: System prompt engineering
# Modify the system_message parameter below

response_a = chat(
    "What is the difference between RAM and a hard drive?",
    system_message="You are a helpful assistant.",  # <-- change this
    max_new_tokens=150,
    temperature=0.7,
)
print(response_a)

In [ ]:
# Part B: Length control
# Run once with max_new_tokens=50 and once with max_new_tokens=300

question = "Explain how gradient descent works."

for token_limit in [50, 300]:
    print(f"\n--- max_new_tokens={token_limit} ---")
    response = chat(question, max_new_tokens=token_limit, temperature=0.3)
    print(response)
    print(f"(Response length: {len(response.split())} words)")

In [ ]:
# Part C: Find a failure mode
# Try different prompts to find where the model struggles

hard_prompt = """I have a 3x3 grid. I place a token at position (row=1, col=1) which is the top-left corner.
I move it right 2 spaces, then down 1 space, then left 1 space.
What position (row, col) is the token at now, and is it still inside the grid?"""

response_c = chat(
    hard_prompt,
    system_message="You are a careful reasoner. Think step by step.",
    max_new_tokens=200,
    temperature=0.1,
)
print(response_c)

print("\n--- Expected ---")
print("Start: (1,1). After right 2: (1,3). After down 1: (2,3). After left 1: (2,2).")
print("Final position: row=2, col=2. Still inside a 3x3 grid.")
print("\nNote any differences between the model's answer and the expected answer.")

## Summary

What you did in this notebook:

- Loaded a 1.5B parameter instruction-tuned model with 4-bit NF4 quantization using `bitsandbytes`.
- Used `device_map="auto"` to automatically place model layers on the GPU.
- Wrapped the model in a `pipeline` and formatted prompts with `apply_chat_template`.
- Ran the model on three different task types and observed where it succeeds and where it struggles.

**Key takeaways:**

- Model size matters: 1.5B models are good for factual recall and basic code, but struggle with complex multi-step reasoning. A 7B model with the same quantization would fit on a T4 and do noticeably better.
- Temperature controls the randomness-vs-determinism tradeoff. Use low temperature (0.1-0.3) for factual/code tasks, higher (0.7-1.0) for creative tasks.
- The instruction-following format matters: passing a raw string to an instruct model often produces poor output. Always use `apply_chat_template` for instruct models.
- 4-bit quantization makes it practical to run models that would otherwise be too large, with modest quality loss.